# B5 - Conformal calibration (CQR + ACP)

The classical / neural models in B3/B4 produce raw quantile predictions
with no guarantee of calibration. Two wrappers fix that:

* **`ConformalQuantileRegressor`** (CQR, Romano-Patterson-Candès 2019).
  Computes nonconformity scores on a held-out calibration set and
  expands the interval by their (1-α) quantile to guarantee marginal
  coverage.
* **`AdaptiveConformalCalibrator`** (Gibbs-Candès 2021). Online update of
  a single δ each step; gives long-run conditional coverage under
  distribution shift.

We apply CQR to the best classical model (CatBoost) from B3.

In [ ]:
import sys
from pathlib import Path

REPO_ROOT = Path.cwd().resolve()
while not (REPO_ROOT / 'src').exists() and REPO_ROOT != REPO_ROOT.parent:
    REPO_ROOT = REPO_ROOT.parent
sys.path.insert(0, str(REPO_ROOT / 'src'))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option('display.max_columns', 50)
pd.set_option('display.width', 180)

## Build dataset (same recipe as B3)

In [ ]:
from data.loaders import load_split
from module_b.features import prepare_supervised, HORIZON_COL
from module_b.features import build_features

train_df = load_split('train')
val_df = load_split('val')
full = pd.concat([train_df, val_df])

feat = build_features(full, bundles=('calendar', 'lags', 'fundamentals',
                                     'spike', 'regime', 'weather'))
past_cols = [c for c in feat.columns if
             c.startswith('price_lag') or c.startswith('price_rmean')
             or c.startswith('price_rstd')
             or c in ('residual_load', 'renewable_penetration',
                      'clean_spark_anchor', 'clean_dark_anchor',
                      'is_high_residual_load', 'is_renewable_scarcity',
                      'crisis_2022_flag')]
future_cols = ['hour_sin', 'hour_cos', 'dow_sin', 'dow_cos',
               'is_weekend', 'is_holiday_DE',
               'weather_mean_wind_speed_100m', 'weather_mean_shortwave_radiation']
X, y = prepare_supervised(feat, horizons=range(1, 25),
                          past_cols=[c for c in past_cols if c in feat.columns],
                          future_cols=[c for c in future_cols if c in feat.columns])

# Split train into proper-train (fit) and calibration (CQR).
is_2023 = (X['origin_ts'] >= pd.Timestamp('2023-01-01', tz='UTC')) & (X['origin_ts'] < val_df.index.min())
is_pre_2023 = X['origin_ts'] < pd.Timestamp('2023-01-01', tz='UTC')
is_val = X['origin_ts'] >= val_df.index.min()
X_fit, y_fit = X.loc[is_pre_2023], y.loc[is_pre_2023]    # fit
X_cal, y_cal = X.loc[is_2023], y.loc[is_2023]            # calibrate
X_test, y_test = X.loc[is_val], y.loc[is_val]            # evaluate (2024 val)
print(f'fit={len(X_fit):,}, cal={len(X_cal):,}, test={len(X_test):,}')

## Fit + CQR-calibrate - one cell per base model

Each block below fits one base model on the pre-2023 chunk, calibrates with CQR on
2023, evaluates 80% coverage on the 2024 val set, and saves the fitted base + the
CQR state to per-model paths under `outputs/`. If one cell hangs, interrupt it and
keep the others' artifacts.


In [ ]:
from module_b.models import (
    CatBoostQuantileForecaster, LightGBMQuantileForecaster,
)
from module_b.evaluation import ConformalQuantileRegressor, coverage

outputs_dir = REPO_ROOT / 'notebooks' / 'module_b' / 'outputs'
outputs_dir.mkdir(exist_ok=True)

base_models = {}   # name -> fitted base model
cqr_models  = {}   # name -> calibrated CQR
predictions = {}   # name -> (q_base, q_cal) on X_test

def fit_and_calibrate(name, factory, alpha=0.2):
    base = factory()
    base.fit(X_fit, y_fit, X_val=X_cal, y_val=y_cal)
    q_base = base.predict_quantiles(X_test)
    cqr = ConformalQuantileRegressor(base=base, alpha=alpha)
    cqr.calibrate(X_cal, y_cal)
    q_cal = cqr.predict_quantiles(X_test)
    base_models[name] = base
    cqr_models[name]  = cqr
    predictions[name] = (q_base, q_cal)
    cov_base = coverage(y_test, q_base['q10'], q_base['q90'])
    cov_cal  = coverage(y_test, q_cal['q10'],  q_cal['q90'])
    print(f'[{name}] 80% coverage  base={cov_base:.3f}  cqr={cov_cal:.3f}  '
          f'delta={cqr.delta:.3f} \u20ac/MWh')
    base.save(outputs_dir / f'B5_{name}_base')
    cqr.save(outputs_dir / f'B5_{name}_cqr')
    return base, cqr


### CatBoost base + CQR

In [ ]:
fit_and_calibrate(
    'catboost',
    lambda: CatBoostQuantileForecaster(iterations=400, early_stopping_rounds=30),
)


### LightGBM base + CQR

In [ ]:
fit_and_calibrate(
    'lightgbm',
    lambda: LightGBMQuantileForecaster(num_boost_round=400, early_stopping_rounds=30),
)


## Full training + CQR \u2014 class defaults

Same factories but with no `iterations` / `num_boost_round` cap. Saves under
names `<model>_full` so the light artifacts above are preserved.

**Expected wall-time on M3 Pro:** CatBoost \u2248 15\u201330 min, LightGBM \u2248 15\u201330 min.


### CatBoost - full

In [ ]:
fit_and_calibrate(
    'catboost_full',
    # Per-quantile mode is the new default; cap iterations at 2000 / ES=30 so
    # the 3 per-quantile boosters fit in a reasonable time.
    lambda: CatBoostQuantileForecaster(iterations=2000, early_stopping_rounds=30),
)


### LightGBM - full

In [ ]:
fit_and_calibrate('lightgbm_full', lambda: LightGBMQuantileForecaster())


## Pick a model for the plot + coverage breakdown below

The downstream plot and per-horizon coverage table use a single `base`/`cqr`
pair. Pick whichever base model you actually trained successfully.


In [ ]:
name = 'catboost'   # change to 'catboost'/'lightgbm' as needed
base, cqr = base_models[name], cqr_models[name]
q_base, q_cal = predictions[name]
print(f'Selected {name}')


## Visualise a sample week

In [ ]:
import matplotlib.dates as mdates

test_h1 = (X_test[HORIZON_COL] == 1)
win_mask = (X_test['target_ts'] >= pd.Timestamp('2024-06-01', tz='UTC')) & \
           (X_test['target_ts'] <= pd.Timestamp('2024-06-08', tz='UTC')) & test_h1
ts = X_test.loc[win_mask, 'target_ts'].to_numpy()
qb = q_base.loc[win_mask].reset_index(drop=True)
qc = q_cal.loc[win_mask].reset_index(drop=True)
yt = y_test.loc[win_mask].to_numpy()

fig, axes = plt.subplots(2, 1, figsize=(12, 6), sharex=True)
for ax, q, title in zip(axes, [qb, qc], ['Base', 'CQR-calibrated']):
    ax.fill_between(ts, q['q10'], q['q90'], alpha=0.3, color='C0', label='80% PI')
    ax.plot(ts, q['q50'], color='C0', label='median')
    ax.plot(ts, yt, color='k', alpha=0.6, label='actual')
    ax.set_title(title); ax.legend(loc='upper right'); ax.set_ylabel('€/MWh')
plt.suptitle('Day-ahead price interval for horizon=1, week of 2024-06-01')
plt.tight_layout()
plt.show()

## Coverage by horizon group

In [ ]:
from module_b.features import HORIZON_RANGES, HorizonGroup

rows = []
for group in HorizonGroup:
    mask = X_test[HORIZON_COL].isin(HORIZON_RANGES[group]).to_numpy()
    rows.append({
        'horizon_group': group.value,
        'coverage_base': coverage(y_test.to_numpy()[mask],
                                   q_base['q10'].to_numpy()[mask],
                                   q_base['q90'].to_numpy()[mask]),
        'coverage_cqr': coverage(y_test.to_numpy()[mask],
                                  q_cal['q10'].to_numpy()[mask],
                                  q_cal['q90'].to_numpy()[mask]),
    })
pd.DataFrame(rows).round(3)

## Save the calibrated model artifacts

In [ ]:
print('Saved artifacts in', outputs_dir)
for name in base_models:
    print(f'  {name}: B5_{name}_base, B5_{name}_cqr')
